# EE/CS 148B HW4 — Diffusion Models
**Run on Colab Pro+ with A100 GPU for Parts 5–7.**

## 0. Setup

In [ ]:
# Mount Google Drive to persist checkpoints
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# Clone the repo (replace with your GitHub URL or upload files)
if not os.path.exists('/content/hw4'):
    !git clone https://github.com/caltech-eecs148b/hw4 /content/hw4

os.chdir('/content/hw4')
!pip install -e . -q
!pip install torch-fidelity -q

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Symlink runs/ directory to Drive for persistence
DRIVE_RUNS = '/content/drive/MyDrive/cs148_hw4_runs'
os.makedirs(DRIVE_RUNS, exist_ok=True)
if not os.path.islink('/content/hw4/runs'):
    os.symlink(DRIVE_RUNS, '/content/hw4/runs')

## 1. Verify Tests Pass

In [ ]:
!python -m pytest tests/ -v

## 2. Problem 1.8 — Coefficient Plot

In [ ]:
!python scripts/plot_coefficient.py --T 1000 --beta_start 1e-4 --beta_end 0.02 \
    --out coefficient_plot.png

from IPython.display import Image
Image('coefficient_plot.png')

## 3. Part 5: VP Score Model on FashionMNIST

### 3.1 Dataset Visualization (Problem 5.C.i)

In [ ]:
import matplotlib.pyplot as plt
import torchvision
from torchvision import datasets, transforms

FASHION_CLASSES = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])
ds = datasets.FashionMNIST('data', train=True, download=True, transform=tf)

# Sample 64 images
loader = torch.utils.data.DataLoader(ds, batch_size=64, shuffle=True)
imgs, labels = next(iter(loader))

grid = torchvision.utils.make_grid(imgs * 0.5 + 0.5, nrow=8)
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(grid.permute(1, 2, 0), cmap='gray')
ax.axis('off')
ax.set_title('FashionMNIST — 64 training samples (28×28 grayscale, 10 classes)')
plt.tight_layout()
plt.savefig('fashionmnist_samples.png', dpi=150)
plt.show()
print('Classes:', FASHION_CLASSES)

### 3.2 Train VP Score Model (Problem 5.C.ii)

In [ ]:
# Train with beta_min=0.01, beta_max=5.0 for 50 epochs
!python scripts/train_vp.py \
    --beta_min 0.01 --beta_max 5.0 \
    --epochs 50 --lr 1e-4 --batch_size 128 \
    --save_dir runs/vp \
    --patience 10 \
    --device cuda

In [ ]:
# Plot training curves (log scale)
import numpy as np

train_losses = np.load('runs/vp/train_losses.npy')
val_losses   = np.load('runs/vp/val_losses.npy')

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(train_losses, label='train')
ax.semilogy(val_losses,   label='val')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (log scale)')
ax.set_title('VP Score Model — Training & Validation Loss')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.savefig('vp_loss_curve.png', dpi=150)
plt.show()

### 3.3 EM Samples (Problem 5.C.iii)

In [ ]:
!python scripts/sample.py --method em \
    --checkpoint runs/vp/best.pt \
    --beta_min 0.01 --beta_max 5.0 \
    --num_steps 1000 --n_samples 64 \
    --out samples_em_1000.png --device cuda

Image('samples_em_1000.png')

### 3.4 PC Samples (Problem 5.C.iv)

In [ ]:
# PC with 1 corrector step
!python scripts/sample.py --method pc \
    --checkpoint runs/vp/best.pt \
    --beta_min 0.01 --beta_max 5.0 \
    --num_steps 1000 --n_corrector 1 --n_samples 64 \
    --out samples_pc_K1.png --device cuda

Image('samples_pc_K1.png')

In [ ]:
# PC with 3 corrector steps
!python scripts/sample.py --method pc \
    --checkpoint runs/vp/best.pt \
    --beta_min 0.01 --beta_max 5.0 \
    --num_steps 1000 --n_corrector 3 --n_samples 64 \
    --out samples_pc_K3.png --device cuda

Image('samples_pc_K3.png')

## 4. Part 6: Rectified Flow on FashionMNIST

### 4.1 Train Rectified Flow (Problem 6.A)

In [ ]:
!python scripts/train_rectflow.py \
    --epochs 50 --lr 1e-4 --batch_size 128 \
    --save_dir runs/rectflow \
    --device cuda

In [ ]:
# Combined loss curves (VP vs Rectified Flow)
vp_train = np.load('runs/vp/train_losses.npy')
rf_train = np.load('runs/rectflow/train_losses.npy')

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(vp_train, label='VP Score Model')
ax.semilogy(rf_train, label='Rectified Flow')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (log scale)')
ax.set_title('Training Loss: VP vs Rectified Flow')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.savefig('combined_loss_curve.png', dpi=150)
plt.show()

### 4.2 Rectified Flow Euler Samples (Problem 6.B)

In [ ]:
for steps in [1, 5, 10, 50, 100]:
    out = f'samples_rf_{steps}.png'
    !python scripts/sample.py --method rectflow \
        --checkpoint runs/rectflow/best.pt \
        --num_steps {steps} --n_samples 64 \
        --out {out} --device cuda
    display(Image(out))

### 4.3 KID Evaluation Table (Problem 6.B)

In [ ]:
!python scripts/eval_kid.py \
    --vp_checkpoint runs/vp/best.pt \
    --rf_checkpoint runs/rectflow/best.pt \
    --beta_min 0.01 --beta_max 5.0 \
    --n_samples 1000 --device cuda

### 4.4 Reflow (Problem 6.C)

In [ ]:
# Generate 50k reflow pairs and retrain for 20 epochs
!python scripts/train_rectflow.py \
    --reflow \
    --checkpoint runs/rectflow/best.pt \
    --n_reflow_pairs 50000 \
    --reflow_steps 100 \
    --epochs 20 \
    --save_dir runs/rectflow_reflow \
    --device cuda

In [ ]:
# One-step generation after reflow
!python scripts/sample.py --method rectflow \
    --checkpoint runs/rectflow_reflow/best.pt \
    --num_steps 1 --n_samples 64 \
    --out samples_reflow_1step.png --device cuda

Image('samples_reflow_1step.png')

### 4.5 Side-by-Side Qualitative Grid (Problem 6.D)

In [ ]:
!python scripts/sample.py --method all \
    --vp_checkpoint runs/vp/best.pt \
    --rf_checkpoint runs/rectflow/best.pt \
    --reflow_checkpoint runs/rectflow_reflow/best.pt \
    --beta_min 0.01 --beta_max 5.0 \
    --seed 42 \
    --out comparison_grid.png \
    --device cuda

Image('comparison_grid.png')

## 5. Part 7: Guided Diffusion on 256×256 Images

In [ ]:
# Clone guided-diffusion and install
if not os.path.exists('/content/guided-diffusion'):
    !git clone https://github.com/openai/guided-diffusion /content/guided-diffusion

%cd /content/guided-diffusion
!pip install -e . -q

# Download model weights (from OpenAI model card)
!mkdir -p models
# Run these manually or provide your own download links
print('Download 256x256_diffusion_uncond.pt and 256x256_classifier.pt to /content/guided-diffusion/models/')

### 5.1 Problem 7.1: Unconditional Generation

In [ ]:
%cd /content/guided-diffusion

MODEL_FLAGS = '--attention_resolutions 32,16,8 --class_cond False --diffusion_steps 1000 \
--image_size 256 --learn_sigma True --noise_schedule linear --num_channels 256 \
--num_head_channels 64 --num_res_blocks 2 --resblock_updown True --use_fp16 True \
--use_scale_shift_norm True'

SAMPLE_FLAGS = '--batch_size 8 --num_samples 8 --timestep_respacing 250'

!OPENAI_LOGDIR=./out_7_1 python scripts/image_sample.py \
    --model_path models/256x256_diffusion_uncond.pt \
    {MODEL_FLAGS} {SAMPLE_FLAGS}

In [ ]:
%cd /content/hw4
import glob

npz_files = glob.glob('/content/guided-diffusion/out_7_1/samples_*.npz')
!python scripts/guided_diffusion_experiments.py --task 7_1 \
    --npz {npz_files[0]} --out 7_1_unconditional.png

Image('7_1_unconditional.png')

### 5.2 Problem 7.2: Progressive Generation

In [ ]:
%cd /content/guided-diffusion

# Use progress_interval or save intermediate states — 
# guided-diffusion supports --use_ddim for fewer steps to get intermediates
# We sample with 250 steps and save every ~30 steps
SAMPLE_FLAGS_PROG = '--batch_size 1 --num_samples 1 --timestep_respacing 250 --use_ddim True'

# For progressive visualization we run 8 separate short runs
for i, steps in enumerate([1, 35, 70, 105, 140, 175, 210, 250]):
    ts = str(steps)
    !OPENAI_LOGDIR=./out_prog_{ts} python scripts/image_sample.py \
        --model_path models/256x256_diffusion_uncond.pt \
        {MODEL_FLAGS} \
        --batch_size 1 --num_samples 1 \
        --timestep_respacing {ts} --use_ddim True

In [ ]:
%cd /content/hw4
prog_npzs = [f'/content/guided-diffusion/out_prog_{s}/samples_*.npz'
             for s in [1, 35, 70, 105, 140, 175, 210, 250]]
prog_npzs_resolved = [glob.glob(p)[0] for p in prog_npzs]

!python scripts/guided_diffusion_experiments.py --task 7_2 \
    --npz {' '.join(prog_npzs_resolved)} --out 7_2_progressive.png

Image('7_2_progressive.png')

### 5.3 Problem 7.3: Noise Interpolation

In [ ]:
# We generate 8 samples with interpolated noise manually using the guided-diffusion API
%cd /content/guided-diffusion
import sys
sys.path.insert(0, '.')

import numpy as np
import torch
from guided_diffusion.script_util import (
    model_and_diffusion_defaults,
    create_model_and_diffusion,
    args_to_dict,
)

def load_model_256():
    from argparse import Namespace
    args = Namespace(
        attention_resolutions='32,16,8', class_cond=False, diffusion_steps=1000,
        image_size=256, learn_sigma=True, noise_schedule='linear', num_channels=256,
        num_head_channels=64, num_res_blocks=2, resblock_updown=True, use_fp16=True,
        use_scale_shift_norm=True, dropout=0.0, timestep_respacing='250',
        rescale_timesteps=False, rescale_learned_sigmas=False,
        use_kl=False, predict_xstart=False, learn_variance=False,
    )
    model, diffusion = create_model_and_diffusion(
        **args_to_dict(args, model_and_diffusion_defaults().keys())
    )
    model.load_state_dict(torch.load('models/256x256_diffusion_uncond.pt', map_location='cpu'))
    model.cuda().eval()
    return model, diffusion

model, diffusion = load_model_256()

In [ ]:
# Generate 8 interpolated samples
torch.manual_seed(1337)
z0 = torch.randn(1, 3, 256, 256).cuda()
torch.manual_seed(9999)
z7 = torch.randn(1, 3, 256, 256).cuda()

all_images = []
for i in range(8):
    alpha = i / 7.0
    z_i = (1 - alpha) * z0 + alpha * z7
    sample = diffusion.p_sample_loop(
        model,
        (1, 3, 256, 256),
        noise=z_i,
        clip_denoised=True,
        progress=False,
    )
    img = ((sample + 1) * 127.5).clamp(0, 255).to(torch.uint8)
    all_images.append(img.cpu().numpy().transpose(0, 2, 3, 1))

arr = np.concatenate(all_images, axis=0)  # (8, 256, 256, 3)
np.savez('/content/guided-diffusion/interp_samples.npz', arr)
print('Generated interpolation samples.')

In [ ]:
%cd /content/hw4
!python scripts/guided_diffusion_experiments.py --task 7_3 \
    --npz /content/guided-diffusion/interp_samples.npz --out 7_3_interpolation.png

Image('7_3_interpolation.png')

### 5.4 Problem 7.4: Conditional Generation

In [ ]:
%cd /content/guided-diffusion

MODEL_FLAGS_COND = '--attention_resolutions 32,16,8 --class_cond True --diffusion_steps 1000 \
--image_size 256 --learn_sigma True --noise_schedule linear --num_channels 256 \
--num_head_channels 64 --num_res_blocks 2 --resblock_updown True --use_fp16 True \
--use_scale_shift_norm True'

CLASSIFIER_FLAGS = '--classifier_scale 1.0 --classifier_path models/256x256_classifier.pt \
--classifier_depth 2 --classifier_width 128 --classifier_attention_resolutions 32,16,8 \
--classifier_use_scale_shift_norm True --classifier_pool attention'

# Generate 8 images conditioned on 8 random ImageNet classes
import random
random.seed(42)
classes = random.sample(range(1000), 8)
print('Classes:', classes)

!OPENAI_LOGDIR=./out_7_4 python scripts/classifier_sample.py \
    --model_path models/256x256_diffusion_uncond.pt \
    {MODEL_FLAGS_COND} {CLASSIFIER_FLAGS} \
    --batch_size 8 --num_samples 8 --timestep_respacing 250

In [ ]:
%cd /content/hw4
npz_7_4 = glob.glob('/content/guided-diffusion/out_7_4/samples_*.npz')[0]
!python scripts/guided_diffusion_experiments.py --task 7_4 \
    --npz {npz_7_4} --out 7_4_conditional.png

Image('7_4_conditional.png')

### 5.5 Problem 7.5: Classifier Scale Sweep

In [ ]:
%cd /content/guided-diffusion

scale_values = [0.5, 1.0, 2.0, 4.0, 8.0, 16.0, 32.0, 64.0]
npz_paths = []

for scale in scale_values:
    out_dir = f'./out_scale_{scale}'
    CLASSIFIER_FLAGS_S = f'--classifier_scale {scale} --classifier_path models/256x256_classifier.pt \
--classifier_depth 2 --classifier_width 128 --classifier_attention_resolutions 32,16,8 \
--classifier_use_scale_shift_norm True --classifier_pool attention'
    !OPENAI_LOGDIR={out_dir} python scripts/classifier_sample.py \
        --model_path models/256x256_diffusion_uncond.pt \
        {MODEL_FLAGS_COND} {CLASSIFIER_FLAGS_S} \
        --batch_size 2 --num_samples 2 --timestep_respacing 250
    found = glob.glob(f'{out_dir}/samples_*.npz')
    if found:
        npz_paths.append(found[0])

In [ ]:
%cd /content/hw4
!python scripts/guided_diffusion_experiments.py --task 7_5 \
    --npz {' '.join(npz_paths)} --out 7_5_scale_sweep.png

Image('7_5_scale_sweep.png')

## 6. (EC) Inpainting — Problem 5.D

In [ ]:
%cd /content/hw4
import torch
from diffusion.unet import UNet
from diffusion.vp import VPSDE
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

sde = VPSDE(beta_min=0.01, beta_max=5.0)
model = UNet(in_channels=1, base_channels=64).to(device)
model.load_state_dict(torch.load('runs/vp/best.pt', map_location=device))
model.eval()

tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])
ds = datasets.FashionMNIST('data', train=False, download=True, transform=tf)

# Pick 5 clean images
clean_imgs = torch.stack([ds[i][0] for i in range(5)]).to(device)  # (5, 1, 28, 28)

# Create corruption mask: 75% of pixels set to 0
torch.manual_seed(0)
mask = (torch.rand(5, 1, 28, 28, device=device) > 0.75).float()
corrupted = clean_imgs * mask

# Inpaint
reconstructed = sde.inpaint(model, corrupted, mask, num_steps=1000, device=device)

def psnr(a, b):
    mse = ((a - b) ** 2).mean()
    return (20 * torch.log10(2.0 / torch.sqrt(mse))).item()

# Plot: 3 rows (clean, corrupted, reconstructed), 5 columns
fig, axes = plt.subplots(3, 5, figsize=(10, 6))
row_labels = ['Clean', 'Corrupted (75%)', 'Reconstructed']
for i in range(5):
    for row, imgs in enumerate([clean_imgs, corrupted, reconstructed]):
        ax = axes[row, i]
        img = imgs[i, 0].cpu().numpy() * 0.5 + 0.5
        ax.imshow(img, cmap='gray', vmin=0, vmax=1)
        ax.axis('off')
        if i == 0:
            ax.set_ylabel(row_labels[row], fontsize=9)
        if row == 2:
            p = psnr(clean_imgs[i], reconstructed[i])
            ax.set_title(f'PSNR={p:.1f}', fontsize=8)

plt.suptitle('Inpainting — 75% corruption (Problem 5.D)', fontsize=11)
plt.tight_layout()
plt.savefig('inpainting_results.png', dpi=150)
plt.show()